# Paper 1 R1 — full table refresh (Option A)

Re-runs the whole GNN suite once in a pinned environment so every table number is internally consistent and reproducible, and adds AUPRC where reviewers asked.

Set GPU runtime (A100 preferred). Run cells top-to-bottom. Each cell copies its result JSON into `experiments_R1/results/`. Cells 5–6 (label efficiency) are the slow ones — everything before them is already saved, so a timeout there loses nothing.

| Cell | Tables | Cost |
|---|---|---|
| 2 | 4, 5(Sup), 6, 7-RAS(+AUPRC) | medium |
| 3 | 8, 9 (+AUPRC) | medium |
| 4 | 5 (SSL metadata) | small |
| 5 | 10 RAS-Eval | heavy |
| 6 | 10 ATBench + Combined | heaviest |
| 7 | classical ATBench/Combined (Table 7) | small |

In [ ]:
# Cell 1 - mount, install, pin exact versions
import os, subprocess
from google.colab import drive; drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/mcp-agent-attack-detection'
REVDIR=os.path.join(ROOT,'experiments_R1','results')
assert os.path.isdir(ROOT), f'fix ROOT: {ROOT}'
os.chdir(ROOT)
import torch; print('torch',torch.__version__,'| cuda',torch.version.cuda)
!pip -q install torch_geometric sentence-transformers xgboost
open(os.path.join(REVDIR,'requirements_pinned.txt'),'w').write(subprocess.run(['pip','freeze'],capture_output=True,text=True).stdout)
os.environ.update(PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python',USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='1',R1_RESULTS_DIR=REVDIR)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu', '| versions pinned ->', REVDIR)

In [ ]:
# Cell 2 - Tables 4, 5(Sup), 6 (GNN AUROC) + Table 6/7-RAS with AUPRC
import os
ROOT='/content/drive/MyDrive/mcp-agent-attack-detection'
REVDIR=os.path.join(ROOT,'experiments_R1','results')
os.chdir(ROOT)
os.environ.update(PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python',USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='1',R1_RESULTS_DIR=REVDIR)
!python scripts/graph_supervised.py --mode content  --arch all --pooled --seeds 7,42,123
!cp results_graph_content/results.json {REVDIR}/tab_supervised_content.json
!python scripts/graph_supervised.py --mode metadata --arch all --pooled --seeds 7,42,123
!cp results_graph_metadata/results.json {REVDIR}/tab_supervised_metadata.json
!python experiments_R1/e_d_faithful.py    # Table 6 (faithful AUROC) + Table 7-RAS, both with AUPRC -> e_d_faithful.json
print('cell 2 done')

In [ ]:
# Cell 3 - Tables 8 (per-dataset, has AUPRC) + 9 (per-attack, seed 123)
!python scripts/graph_evaluate.py --mode content --dataset combined --arch sage --pooled --seeds 7,42,123
!cp results_eval_combined_content/results.json {REVDIR}/tab_eval_combined.json
print('cell 3 done')

In [ ]:
# Cell 4 - Table 5 SSL+FT metadata row
!python scripts/graph_ssl.py --mode metadata --pooled --seed 42
!cp results_graph_ssl_metadata/results.json {REVDIR}/tab_ssl_metadata.json
print('cell 4 done')

In [ ]:
# Cell 5 - Table 10 RAS-Eval (label efficiency) + Table 5 SSL+FT content 100% point [HEAVY]
!python scripts/graph_label_efficiency.py --pooled --seed 42
!cp results_label_efficiency/results.json {REVDIR}/tab_labeleff_raseval.json
print('cell 5 done')

In [ ]:
# Cell 6 - Table 10 ATBench + Combined columns (wrapper) [HEAVIEST]
!python experiments_R1/labeleff_multidataset.py --dataset atbench --seed 42
!cp results_label_efficiency_atbench/results.json {REVDIR}/tab_labeleff_atbench.json
!python experiments_R1/labeleff_multidataset.py --dataset combined --seed 42
!cp results_label_efficiency_combined/results.json {REVDIR}/tab_labeleff_combined.json
print('cell 6 done')

In [ ]:
# Cell 7 - Table 7 classical for ATBench + Combined (AUROC/F1/recall/FPR)
!python scripts/classical_baselines.py --dataset atbench
!cp results/classical_baselines_atbench.json {REVDIR}/tab_classical_atbench.json
!python scripts/classical_baselines.py --dataset combined
!cp results/classical_baselines_combined.json {REVDIR}/tab_classical_combined.json
print('all cells done — result JSONs are in', REVDIR)